# 自然言語処理２：大規模言語モデルにおけるTokenizer

文書の意味理解を行うための最初の手続きとしては、「自然言語処理２」で紹介した`janome`のように、品詞を意識した単語分割を行うのが一般的でした。
これは、長きにわたる言語学の歴史で培われた様々な知見を活かせるという意味では大きな利点がありましたが、様々な言語を同じ枠組みで扱えないことや、必ずしも頻繁に現れない単語についても同じ粒度で分割してしまうという欠点がありました。

文書ベクトルでは、異なる単語にはすべて異なる次元を割り当てます。
このとき、「の」のように頻出する単語も１次元割り当て、「駒場」のように（東大以外では）めったに使われない単語も１次元割り当てるのはもったいないと思いませんか？
意味的なまとまりを無視して「駒場」を「駒」と「場」に分けてよければ、どちらもある程度頻繁に現れるものになりそうです。
一方、「今日」は２文字からなる単語ですが、これは頻出するので「今」と「日」に分けなくてもいいでしょう。

このように、頻出する単語は大きく、めったに現れない単語は小さく、というように単語長を調節して切り分けた単語を「サブワード」と呼びます。
確率的に単語境界を決定するため、多言語化が容易で、データサイズも圧縮できることが大きな利点であり、昨今の大規模言語モデル（Large Language Model, LLM）ではこの方法が用いられています。

なお、ここで切り分けられた各単語は、言語学的な意味の「単語」ではないため、このような手続きは tokenizing (トークンにする）という表現が使われます。

## 1. ライブラリのインストール

ここでは、第４回で紹介する汎用自然言語処理モデルBERTで用いられているトークナイザ、WordPieceを使ってトークナイジングを行ってみましょう。
まず、ライブラリをインストールします。

*  `transformers`: BERTで使われる様々な関数がまとめられたパッケージ
*  `fugashi`: 形態素解析器
*  `unidic-lite`, `ipadic`: 日本語辞書

In [ ]:
!pip install transformers fugashi unidic-lite ipadic

`fugashi`は`janome`と同じ形態素解析器です。


In [ ]:
from fugashi import Tagger

tagger = Tagger()
line = '東京大学は欧米諸国の諸制度に倣った、日本国内で初の近代的な大学として設立された。'
for word in tagger.parseToNodeList(line):
    print(word, word.feature.lemma, word.pos, sep='\t')

## 2. 日本語モデルのダウンロード

東北大学が提供している日本語用のモデルをダウンロードして読み込みます。

In [ ]:
from transformers import BertJapaneseTokenizer
model_name = 'cl-tohoku/bert-base-japanese-whole-word-masking'
tokenizer = BertJapaneseTokenizer.from_pretrained(model_name)

## 3. トークナイジングの実行

上で起動したトークナイザにテキストを渡すと、トークンに分割します。

In [ ]:
line = '東京大学は欧米諸国の諸制度に倣った、日本国内で初の近代的な大学として設立された。'
token = tokenizer.tokenize(line)
token

文書解析では、単語に分割するだけではなく、これをダミー変数に変換する必要があります。
このtokenizerが与えられた文をどのようなダミー変数に変換したか確認してみましょう。

In [ ]:
input_ids = tokenizer.encode(line)
input_ids

上の結果と下の結果を見比べてみてください。
`2`は文頭、`3`は文末に割り当てられたダミー変数です。
また、「東京大学」は`4610`、「は」は`9`、「大学」は`396`のようです。
このように、同じモデルを使う場合、同じ単語は同じダミー変数へと変換されます。

In [ ]:
print(tokenizer.tokenize('東京大学は大学だ。'))
print(tokenizer.encode('東京大学は大学だ。'))

## 4. 大規模言語モデルの Multi-lingual Tokenizer

ChatGPTでは、日本語でも英語でも意識せずに入力することができますよね？
最近の大規模言語モデル（Large Language Model, LLM）では、言語に依存せずどのような言語でも１つのモデルで分割することができます。
ChatGPTがどのようにトークナイジングを行っているか、[こちらのサイト](https://platform.openai.com/tokenizer)で確認してみましょう。
英単語は世界的に見て頻出であるため、100トークンで凡そ75単語が表現できます。
一方、和単語は出現頻度が低いことから、「の」のような高頻度文字だと1トークン1文字ですが、「日」は１文字２トークンになり、「音」は１文字３トークンとなります。

ChatGPTのような状態記憶型（stateful）のモデルでは、新たに入力された文に加え、それまでに入力された文書を記憶して応答しますが、これは無限ではなく、ChatGPT3で4097トークンです。
英語と日本語のトークンの粒度を考えれば、英語の方がより長い文脈を記憶して対話できるということになるでしょう。

